# Visualize và Phân Tích Kết Quả `stats_analysis`

Notebook này đọc trực tiếp các CSV trong `output/stats_analysis` để trực quan hóa và phân tích kết quả đánh giá mô hình biến động tài chính.

**Mục tiêu khoa học:** kiểm tra quan hệ giữa độ chính xác dự báo volatility (`MSE`, `MAE`, `QLIKE`) và chất lượng kiểm soát rủi ro (`Violation Rate`, `Kupiec`, `Christoffersen`, `Pass Rate`).

**Nguyên tắc:** notebook này không chạy lại pipeline tính metrics và không ghi đè CSV đầu vào.

## 1. Load Data

Các bảng đầu vào gồm kết quả chi tiết theo dataset-horizon, kết quả aggregate theo model, pass/fail backtesting cases, và metadata của lần chạy.

In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "output" / "stats_analysis").exists():
    PROJECT_ROOT = Path("D:/UIT_LEARNING_MATERIAL/07.Research/code")

RESULT_DIR = os.environ.get("STATS_ANALYSIS_RESULT_DIR")
DATA_DIR = Path(RESULT_DIR) if RESULT_DIR else PROJECT_ROOT / "output" / "stats_analysis"
FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.6f}".format)

detailed = pd.read_csv(DATA_DIR / "stats_by_dataset_horizon.csv")
aggregate = pd.read_csv(DATA_DIR / "stats_by_model.csv")
pass_cases = pd.read_csv(DATA_DIR / "backtest_pass_cases.csv")
summary = pd.read_csv(DATA_DIR / "stats_run_summary.csv")

def normalize_labels(df):
    out = df.copy()
    out["tier"] = out["tier"].fillna("").replace("", "No Tier")
    out["model_label"] = out["branch"].astype(str) + " / " + out["tier"].astype(str) + " / " + out["model"].astype(str)
    return out

detailed = normalize_labels(detailed)
aggregate = normalize_labels(aggregate)
pass_cases = normalize_labels(pass_cases)

assert detailed.shape[0] == 855, f"Unexpected detailed rows: {detailed.shape[0]}"
assert aggregate.shape[0] == 19, f"Unexpected aggregate rows: {aggregate.shape[0]}"
assert pass_cases.shape[0] == 855, f"Unexpected pass-case rows: {pass_cases.shape[0]}"

TARGET_ALPHA = float(summary.loc[0, "alpha"]) if "alpha" in summary.columns else 0.05
print(f"Data directory: {DATA_DIR}")
print(f"VaR alpha: {TARGET_ALPHA:.2%}")
print(f"Seaborn available: {sns is not None}")
print(f"Detailed: {detailed.shape}, Aggregate: {aggregate.shape}, Pass cases: {pass_cases.shape}")


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\UIT_LEARNING_MATERIAL\\07.Research\\code\\output\\stats_analysis\\stats_by_dataset_horizon.csv'

### Helper Functions

Các hàm dưới đây giúp chuẩn hóa biểu đồ, lưu figure, và fallback khi không có `seaborn`.

In [ ]:
def save_current_figure(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path

def short_label(label, max_len=42):
    label = str(label)
    return label if len(label) <= max_len else label[: max_len - 3] + "..."

def add_value_labels(ax, fmt="{:.3f}"):
    for patch in ax.patches:
        width = patch.get_width()
        if np.isfinite(width):
            ax.text(width, patch.get_y() + patch.get_height() / 2, " " + fmt.format(width), va="center", fontsize=8)

def heatmap_plot(pivot, title, filename, cmap="viridis", fmt=".3f"):
    fig, ax = plt.subplots(figsize=(12, max(5, 0.38 * len(pivot))))
    if sns is not None:
        sns.heatmap(pivot, annot=True, fmt=fmt, cmap=cmap, linewidths=0.3, ax=ax)
    else:
        data = pivot.to_numpy(dtype=float)
        im = ax.imshow(data, aspect="auto", cmap=cmap)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        fig.colorbar(im, ax=ax)
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                if np.isfinite(data[i, j]):
                    ax.text(j, i, format(data[i, j], fmt), ha="center", va="center", color="white", fontsize=8)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=0)
    ax.tick_params(axis="y", labelsize=8)
    saved = save_current_figure(filename)
    plt.show()
    return saved


## 2. Data Quality and Coverage

Bảng này kiểm tra phạm vi benchmark và số dòng hợp lệ cho forecast/risk metrics. Đây là bước quan trọng trước khi diễn giải ranking mô hình.

In [ ]:
coverage = pd.DataFrame([
    {"metric": "Datasets", "value": detailed["dataset"].nunique(), "detail": ", ".join(sorted(detailed["dataset"].astype(str).unique()))},
    {"metric": "Horizons", "value": detailed["horizon"].nunique(), "detail": ", ".join(map(str, sorted(detailed["horizon"].unique())))},
    {"metric": "Raw model names", "value": detailed["model"].nunique(), "detail": ", ".join(sorted(detailed["model"].astype(str).unique()))},
    {"metric": "Model groups", "value": aggregate["model_label"].nunique(), "detail": "branch / tier / model"},
    {"metric": "Detailed cases", "value": len(detailed), "detail": "model group x dataset x horizon"},
])

display(coverage)
display(summary.T.rename(columns={0: "value"}))


**Nhận xét.** Dữ liệu sau thống kê bao phủ đầy đủ 9 thị trường và 5 horizon. File merged hiện có đủ `log_return`; các dòng forecast invalid còn lại đến từ `true_volatility` bị thiếu, chủ yếu ở branch GARCH. Forecast metrics vẫn dùng các dòng có đủ `true_volatility` và `predict_volatility`.

## 3. Aggregate Model Ranking

`pass_rate` càng cao càng tốt cho quản trị rủi ro. `QLIKE`, `MSE`, `MAE` càng thấp càng tốt cho dự báo volatility.

In [ ]:
ranking_cols = ["model_label", "cases", "valid_risk_cases", "passed_cases", "pass_rate", "qlike", "mse", "mae", "violation_rate", "kupiec_p", "lr_ind_p"]

rank_by_pass = aggregate.sort_values(["pass_rate", "qlike"], ascending=[False, True])[ranking_cols].reset_index(drop=True)
rank_by_qlike = aggregate.sort_values(["qlike", "pass_rate"], ascending=[True, False])[ranking_cols].reset_index(drop=True)
rank_by_mse = aggregate.sort_values(["mse", "pass_rate"], ascending=[True, False])[ranking_cols].reset_index(drop=True)
rank_by_mae = aggregate.sort_values(["mae", "pass_rate"], ascending=[True, False])[ranking_cols].reset_index(drop=True)

display(rank_by_pass.style.set_caption("Ranking by Pass Rate"))
display(rank_by_qlike.head(10).style.set_caption("Top 10 by QLIKE"))
display(rank_by_mse.head(10).style.set_caption("Top 10 by MSE"))
display(rank_by_mae.head(10).style.set_caption("Top 10 by MAE"))


**Nhận xét.** Ranking theo `pass_rate` và ranking theo forecasting loss không nhất thiết giống nhau. Đây là tín hiệu chính cho thấy mô hình dự báo volatility tốt chưa chắc đã tạo VaR được hiệu chỉnh tốt.

## 4. Forecasting Accuracy Visualization

Các biểu đồ này tập trung vào `QLIKE`, `MSE`, và `MAE`. Với `MSE/MAE`, dùng log scale vì có outlier lớn làm méo trục tuyến tính.

In [ ]:
plot_df = aggregate.sort_values("qlike", ascending=True).copy()
plot_df["short_label"] = plot_df["model_label"].map(short_label)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(plot_df["short_label"], plot_df["qlike"], color="#2f6f7e")
ax.invert_yaxis()
ax.set_xlabel("QLIKE (lower is better)")
ax.set_title("Forecasting Accuracy Ranking by QLIKE")
add_value_labels(ax, "{:.3f}")
save_current_figure("qlike_ranking.png")
plt.show()


In [ ]:
plot_df = aggregate.sort_values("mse", ascending=True).copy()
plot_df["short_label"] = plot_df["model_label"].map(short_label)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
axes[0].barh(plot_df["short_label"], plot_df["mse"], color="#455a64")
axes[0].set_xscale("log")
axes[0].invert_yaxis()
axes[0].set_title("MSE by Model Group (log scale)")
axes[0].set_xlabel("MSE")

plot_df_mae = aggregate.sort_values("mae", ascending=True).copy()
plot_df_mae["short_label"] = plot_df_mae["model_label"].map(short_label)
axes[1].barh(plot_df_mae["short_label"], plot_df_mae["mae"], color="#7a5c3e")
axes[1].set_xscale("log")
axes[1].invert_yaxis()
axes[1].set_title("MAE by Model Group (log scale)")
axes[1].set_xlabel("MAE")

save_current_figure("mse_mae_log_ranking.png")
plt.show()


**Nhận xét.** `QLIKE` giúp đánh giá volatility forecast trong bối cảnh target dương và nhiễu. Khi `MSE/MAE` có outlier, log scale giúp nhìn được cả nhóm mô hình tốt và mô hình lệch lớn mà không mất thông tin.

## 5. Risk Backtesting Visualization

Phần này phân tích hiệu quả VaR backtesting qua `pass_rate`, `violation_rate`, và số case pass/fail của Kupiec và Christoffersen.

In [ ]:
plot_df = aggregate.sort_values("pass_rate", ascending=True).copy()
plot_df["short_label"] = plot_df["model_label"].map(short_label)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(plot_df["short_label"], plot_df["pass_rate"], color="#2d7d46")
ax.set_xlabel("Pass Rate")
ax.set_xlim(0, max(0.35, plot_df["pass_rate"].max() * 1.2))
ax.set_title("VaR Backtesting Pass Rate by Model Group")
add_value_labels(ax, "{:.3f}")
save_current_figure("pass_rate_ranking.png")
plt.show()


In [ ]:
pass_summary = pass_cases.groupby("model_label", as_index=False).agg(
    cases=("backtest_pass", "size"),
    kupiec_passes=("kupiec_pass", "sum"),
    independence_passes=("independence_pass", "sum"),
    joint_passes=("backtest_pass", "sum"),
)
pass_summary = pass_summary.sort_values("joint_passes", ascending=False)

plot_df = pass_summary.copy()
plot_df["short_label"] = plot_df["model_label"].map(short_label)
x = np.arange(len(plot_df))
width = 0.27

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width, plot_df["kupiec_passes"], width, label="Kupiec pass", color="#4c78a8")
ax.bar(x, plot_df["independence_passes"], width, label="Independence pass", color="#f58518")
ax.bar(x + width, plot_df["joint_passes"], width, label="Joint pass", color="#54a24b")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["short_label"], rotation=70, ha="right")
ax.set_ylabel("Number of dataset-horizon cases")
ax.set_title("Kupiec, Christoffersen, and Joint Backtest Pass Counts")
ax.legend()
save_current_figure("backtest_pass_counts.png")
plt.show()

display(pass_summary)


**Nhận xét.** Một mô hình có thể vượt qua independence test nhiều lần nhưng vẫn fail Kupiec coverage nếu violation rate lệch xa mức tail đang đánh giá. Do đó joint pass là tiêu chí chặt hơn và phù hợp để tổng hợp risk reliability.

## 6. Accuracy-Risk Trade-off

Biểu đồ scatter dưới đây là phần trung tâm của phân tích: trục x là `QLIKE` thấp hơn tốt hơn, trục y là `pass_rate` cao hơn tốt hơn.

In [ ]:
plot_df = aggregate.dropna(subset=["qlike", "pass_rate"]).copy()

fig, ax = plt.subplots(figsize=(11, 7))
branches = sorted(plot_df["branch"].unique())
colors = dict(zip(branches, ["#4c78a8", "#f58518", "#54a24b", "#b279a2"][: len(branches)]))

for branch, group in plot_df.groupby("branch"):
    ax.scatter(group["qlike"], group["pass_rate"], s=95, alpha=0.82, label=branch, color=colors.get(branch))
    for _, row in group.iterrows():
        ax.annotate(short_label(row["model_label"], 28), (row["qlike"], row["pass_rate"]), xytext=(5, 4), textcoords="offset points", fontsize=8)

ax.set_xlabel("QLIKE (lower is better)")
ax.set_ylabel("Pass Rate (higher is better)")
ax.set_title("Forecasting Accuracy vs Risk Backtesting Reliability")
ax.grid(True, alpha=0.25)
ax.legend(title="Branch")
save_current_figure("qlike_vs_pass_rate.png")
plt.show()

corr = plot_df[["qlike", "mse", "mae", "pass_rate", "violation_rate"]].corr(method="spearman")
display(corr)


**Nhận xét.** Nếu các điểm tốt nhất theo `QLIKE` không nằm ở vùng `pass_rate` cao nhất, điều đó củng cố kết luận rằng tối ưu forecast loss không tự động tối ưu risk calibration.

## 7. Dataset and Horizon Sensitivity

Các heatmap sau xem xét kết quả có ổn định qua horizon và dataset hay không. Đây là phần giúp phát hiện mô hình chỉ tốt ở một vài điều kiện cụ thể.

In [ ]:
pass_by_horizon = detailed.groupby(["model_label", "horizon"], as_index=False)["backtest_pass"].mean()
pivot_pass = pass_by_horizon.pivot(index="model_label", columns="horizon", values="backtest_pass")
pivot_pass = pivot_pass.loc[aggregate.sort_values("pass_rate", ascending=False)["model_label"]]
pivot_pass.index = [short_label(x, 48) for x in pivot_pass.index]

heatmap_plot(pivot_pass, "Pass Rate by Model Group and Horizon", "pass_rate_by_horizon_heatmap.png", cmap="YlGn", fmt=".2f")


In [ ]:
top_models = aggregate.sort_values("pass_rate", ascending=False)["model_label"].head(10).tolist()
violation_by_dataset = detailed[detailed["model_label"].isin(top_models)].groupby(["dataset", "model_label"], as_index=False)["violation_rate"].mean()
pivot_violation = violation_by_dataset.pivot(index="dataset", columns="model_label", values="violation_rate")
pivot_violation = pivot_violation[top_models]
pivot_violation.columns = [short_label(x, 24) for x in pivot_violation.columns]

heatmap_plot(pivot_violation, "Mean Violation Rate by Dataset for Top Pass-Rate Models", "violation_rate_by_dataset_heatmap.png", cmap="magma_r", fmt=".3f")


**Nhận xét.** Heatmap theo horizon cho biết mô hình nào ổn định trên nhiều khoảng dự báo. Heatmap theo dataset cho biết risk calibration có phụ thuộc thị trường hay không.

## 8. Distribution by Model Branch

So sánh theo model family giúp nhìn rõ khác biệt giữa econometric, transformer-based, và foundation/hybrid variants.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if sns is not None:
    sns.boxplot(data=detailed, x="branch", y="qlike", ax=axes[0], color="#9ecae1")
    sns.boxplot(data=detailed, x="branch", y="violation_rate", ax=axes[1], color="#a1d99b")
else:
    branches = sorted(detailed["branch"].unique())
    axes[0].boxplot([detailed.loc[detailed["branch"] == b, "qlike"].dropna() for b in branches], labels=branches)
    axes[1].boxplot([detailed.loc[detailed["branch"] == b, "violation_rate"].dropna() for b in branches], labels=branches)

axes[0].set_title("QLIKE Distribution by Branch")
axes[0].set_ylabel("QLIKE")
axes[1].set_title("Violation Rate Distribution by Branch")
axes[1].set_ylabel("Violation Rate")
axes[1].axhline(TARGET_ALPHA, color="red", linestyle="--", linewidth=1, label=f"Target {TARGET_ALPHA:.0%}")
axes[1].legend()
save_current_figure("branch_metric_distributions.png")
plt.show()


## 9. Case-Level Analysis

Các bảng sau liệt kê những case pass backtest và các case fail mạnh nhất theo Kupiec p-value. Đây là nơi hữu ích để kiểm tra từng dataset-horizon cụ thể.

In [ ]:
case_cols = ["model_label", "dataset", "horizon", "n_risk", "violation_count", "violation_rate", "kupiec_p", "lr_ind_p", "kupiec_pass", "independence_pass", "backtest_pass"]

passed_cases = pass_cases[pass_cases["backtest_pass"]].sort_values(["model_label", "dataset", "horizon"])[case_cols]
weakest_kupiec = pass_cases.sort_values("kupiec_p", ascending=True)[case_cols].head(25)
highest_violation = pass_cases.sort_values("violation_rate", ascending=False)[case_cols].head(25)

display(passed_cases.head(50).style.set_caption("Backtest-passed cases"))
display(weakest_kupiec.style.set_caption("Lowest Kupiec p-value cases"))
display(highest_violation.style.set_caption("Highest violation-rate cases"))


## 10. Key Findings

Các kết luận dưới đây được sinh trực tiếp từ bảng aggregate để tránh ghi tay số liệu.

In [ ]:
best_pass = aggregate.sort_values(["pass_rate", "qlike"], ascending=[False, True]).iloc[0]
best_qlike = aggregate.sort_values(["qlike", "pass_rate"], ascending=[True, False]).iloc[0]
best_mse = aggregate.sort_values(["mse", "pass_rate"], ascending=[True, False]).iloc[0]
best_mae = aggregate.sort_values(["mae", "pass_rate"], ascending=[True, False]).iloc[0]

findings = pd.DataFrame([
    {"claim": "Best risk-control model", "evidence": f"{best_pass['model_label']} has highest pass_rate = {best_pass['pass_rate']:.3f}."},
    {"claim": "Best QLIKE model", "evidence": f"{best_qlike['model_label']} has lowest QLIKE = {best_qlike['qlike']:.3f}."},
    {"claim": "Best MSE model", "evidence": f"{best_mse['model_label']} has lowest MSE = {best_mse['mse']:.3g}."},
    {"claim": "Best MAE model", "evidence": f"{best_mae['model_label']} has lowest MAE = {best_mae['mae']:.3f}."},
    {"claim": "Accuracy-risk mismatch", "evidence": "The best pass-rate model and best QLIKE model are different." if best_pass['model_label'] != best_qlike['model_label'] else "The best pass-rate and QLIKE model are the same in this run."},
])

display(findings)


## 11. Output / Export Notes

Notebook này lưu các figure PNG vào `output/stats_analysis/figures` để có thể chèn vào báo cáo hoặc paper draft. Các CSV gốc không bị ghi đè.

In [ ]:
figure_manifest = pd.DataFrame([
    {"figure": p.name, "path": str(p), "size_bytes": p.stat().st_size}
    for p in sorted(FIG_DIR.glob("*.png"))
])

figure_manifest
